In [ ]:
!pip install groq langchain langsmith langchain-groq

In [ ]:
import os
from groq import Groq
from langchain_groq import ChatGroq

# Set your Groq API key
os.environ["GROQ_API_KEY"] = "YOUR GROQ API KEY"

# Enable LangSmith tracing
os.environ["LANGCHAIN_TRACING_V2"] = "true"

In [ ]:
resume_strong = """
John Doe
Experience: 5 years as a Senior Software Engineer at Tech Corp.
Skills: Python, Java, Cloud Computing (AWS/GCP), Kubernetes.
Tools: Docker, Git, Jenkins, Terraform.
"""

resume_average = """
Jane Smith
Experience: 2 years as a Junior Developer.
Skills: Python, Basic SQL, JavaScript.
Tools: Git, VS Code.
"""

resume_weak = """
Bob Wilson
Experience: Recent graduate with a degree in Marketing.
Skills: Microsoft Office, Social Media Management.
Tools: Canva.
"""

job_description = """
Seeking a Senior Software Engineer with expertise in Python, Cloud infrastructure (AWS), and DevOps tools like Kubernetes and Terraform. Minimum 4 years of experience required.
"""

In [ ]:
import os
api_key = "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"

# Updated to currently supported stable Groq models
llm_extractor = ChatGroq(model="llama-3.1-8b-instant", temperature=0, groq_api_key=api_key)
llm_scorer = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, groq_api_key=api_key)

In [ ]:
from langchain_core.prompts import PromptTemplate

skill_extraction_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
    Extract the following from the resume:
    - Skills
    - Experience
    - Tools

    Resume:
    {resume}

    Output in JSON format:
    {{
      "skills": [],
      "experience": "",
      "tools": []
    }}
    """
)

def extract_resume_info(resume):
    prompt = skill_extraction_prompt.format(resume=resume)
    response = llm_extractor.invoke(prompt)
    return response.content

# Note: Ensure 'resume_strong' is defined before running this
# print(extract_resume_info(resume_strong))

In [ ]:
from langchain_core.prompts import PromptTemplate

matching_prompt = PromptTemplate(
    input_variables=["resume_info", "job_description"],
    template="""
    Compare the candidate's extracted skills and experience with the job description.

    Candidate Info:
    {resume_info}

    Job Description:
    {job_description}

    Output in JSON format:
    {{
      "matched_skills": [],
      "missing_skills": [],
      "fit_score": 0
    }}
    """
)

In [ ]:
explanation_prompt = PromptTemplate(
    input_variables=["resume_info", "job_description", "score_result"],
    template="""
    Based on the following analysis, explain why the candidate received this score.

    Candidate Info:
    {resume_info}

    Job Description:
    {job_description}

    Score Result:
    {score_result}

    Provide a recruiter-style explanation in 3–4 sentences.
    """
)

def explain_score(resume_info, job_description, score_result):
    prompt = explanation_prompt.format(
        resume_info=resume_info,
        job_description=job_description,
        score_result=score_result
    )
    response = llm_scorer.invoke(prompt)
    return response.content

# Example run
score_result = score_resume(resume_info, job_description)
print(explain_score(resume_info, job_description, score_result))


The candidate is an exceptional fit for the Senior Software Engineer role, possessing all the required skills including Python, Cloud infrastructure with AWS, and DevOps tools like Kubernetes and Terraform. With 5 years of experience as a Senior Software Engineer, they exceed the minimum 4 years of experience required for the position. Their skill set and experience align perfectly with the job description, making them a highly suitable candidate for the role. Overall, the candidate's strong technical background and relevant experience earn them a perfect fit score, making them an ideal candidate for the position.


In [ ]:
for resume in [resume_strong, resume_average, resume_weak]:
    info = extract_resume_info(resume)
    score = score_resume(info, job_description)
    explanation = explain_score(info, job_description, score)
    print("\nResume:\n", resume)
    print("Extracted Info:", info)
    print("Score Result:", score)
    print("Explanation:", explanation)



Resume:
 
John Doe
Experience: 5 years as a Senior Software Engineer at Tech Corp. 
Skills: Python, Java, Cloud Computing (AWS/GCP), Kubernetes.
Tools: Docker, Git, Jenkins, Terraform.

Extracted Info: Here's the extracted information in JSON format:

```json
{
  "skills": ["Python", "Java", "Cloud Computing (AWS/GCP)", "Kubernetes"],
  "experience": "5 years as a Senior Software Engineer at Tech Corp.",
  "tools": ["Docker", "Git", "Jenkins", "Terraform"]
}
```

However, if you want the "experience" field to be a more structured object, you could represent it like this:

```json
{
  "skills": ["Python", "Java", "Cloud Computing (AWS/GCP)", "Kubernetes"],
  "experience": {
    "duration": "5 years",
    "position": "Senior Software Engineer",
    "company": "Tech Corp."
  },
  "tools": ["Docker", "Git", "Jenkins", "Terraform"]
}
```
Score Result: To compare the candidate's extracted skills and experience with the job description, we'll follow these steps:

1. **Extract required skills